# Senate Amendments Considered

Amendments agreed to or not agreed to on the Senate floor, dated by their latest action. See the README for definitions.

`fetch_amendments` uses the Congress.gov API, runs headless, and takes 10 to 20 minutes. It needs a free key from https://api.congress.gov/sign-up/, saved in a file named `.congress_api_key` at the top of the repo (or in the `CONGRESS_API_KEY` environment variable).

In [ ]:
import pandas as pd
import amendments_helper

START, END = "01/01/2025", None   # MM/DD/YYYY, inclusive; END = None means today

In [ ]:
df = amendments_helper.fetch_amendments(START, END)
outfile = f"amendments_{pd.Timestamp(START):%Y-%m-%d}_to_{pd.Timestamp(END or 'today'):%Y-%m-%d}.csv"
df.to_csv(outfile, index=False)
print(len(df), "amendments saved to", outfile)
df.head()

## Rows to check by hand
Outcome and Action Type come from the text of the latest action. Rows the text doesn't settle (for example, "Senate vitiated previous adoption of amendment...") are marked `INPUT BY HAND`. They count toward totals but not toward UC, VV, or RCV until they are fixed. Check them on Congress.gov, then record the fix in `OVERRIDES` below.

In [ ]:
df[(df["Outcome"] == "INPUT BY HAND") | (df["Action Type"] == "INPUT BY HAND")]

In [ ]:
# Fixes for rows checked by hand: {("118th", "S.Amdt.3292"): {"Outcome": "A", "Action Type": "UC"}, ...}
OVERRIDES = {}

for (congress, number), fixes in OVERRIDES.items():
    rows = df["Congress"].str.startswith(congress) & (df["S.Amdt.No"] == number)
    for column, value in fixes.items():
        df.loc[rows, column] = value
if OVERRIDES:
    df.to_csv(outfile, index=False)  # save the fixes with the data

## Quarterly counts for the full data tables
Per-quarter counts (not cumulative), saved next to the data as `amendments_summary_{start}_to_{end}.csv`. The published sheet is cumulative within each Congress, except for budget resolutions, so add each quarter to the previous one.

- Majority and minority come from `SENATE_MAJORITY` in `amendments_helper.py`. Add each new Congress to it.
- Independents count with the party they caucus with, which is the Democrats.
- The four "amended" categories are mutually exclusive, as in the methodology. They are checked in this order: amendments to amendments, budget resolutions (a concurrent resolution, Con, whose Measure Title sets forth or establishes the congressional budget), other resolutions (Res), and bills (everything else, including treaties).

In [ ]:
summary = amendments_helper.quarterly_summary(df)
summary.to_csv(outfile.replace("amendments_", "amendments_summary_", 1))
summary